# Chapter 6: Algorithm Chains and Pipelines

## Chapter Summary
Pipelines bind preprocessing and prediction into one estimator. They prevent leakage, make cross-validation correct by construction, and allow joint tuning of transformations and models.

## Theoretical Deep-Dive

If scaling or feature selection sees all observations before cross-validation, validation data influences training and scores become optimistic. A pipeline fits each transformation only on the current training fold, then applies the learned transformation to that fold's validation data.

A pipeline is compositional: each intermediate step implements `fit` and `transform`, while the final step implements `fit` and `predict`. Parameters are addressed with `step__parameter`, enabling a single search over preprocessing and model choices.

**Book coverage:** preprocessing with parameter selection, pipeline construction, grid search, step inspection, and model selection (Chapter 6, printed pp. 305-320).

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline, make_pipeline

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, stratify=data.target, random_state=42)
pipeline = Pipeline([("scale", StandardScaler()), ("svc", SVC(random_state=42))])
pipeline.fit(X_train, y_train)
print(f"Pipeline test accuracy: {pipeline.score(X_test, y_test):.3f}")
print("Named steps:", list(pipeline.named_steps))
print("First three learned means:", pipeline.named_steps["scale"].mean_[:3].round(2))

Pipeline test accuracy: 0.979
Named steps: ['scale', 'svc']
First three learned means: [14.08 19.3  91.59]


In [2]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": [0.001, 0.01, 0.1, "scale"],
}
search = GridSearchCV(pipeline, grid, cv=cv, scoring="roc_auc", n_jobs=1)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print(f"CV ROC-AUC: {search.best_score_:.3f}; test ROC-AUC: {search.score(X_test, y_test):.3f}")

Best parameters: {'svc__C': 10, 'svc__gamma': 0.01}
CV ROC-AUC: 0.997; test ROC-AUC: 0.999


In [3]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectPercentile
from sklearn.linear_model import LogisticRegression

flexible = Pipeline([
    ("scale", StandardScaler()),
    ("reduce", "passthrough"),
    ("model", LogisticRegression(max_iter=3000)),
])
parameter_options = [
    {"reduce": [PCA(random_state=42)], "reduce__n_components": [5, 10, 15], "model__C": [0.1, 1, 10]},
    {"reduce": [SelectPercentile()], "reduce__percentile": [30, 60, 100], "model__C": [0.1, 1, 10]},
]
joint_search = GridSearchCV(flexible, parameter_options, cv=cv, scoring="accuracy", n_jobs=1)
joint_search.fit(X_train, y_train)
print("Best joint configuration:", joint_search.best_params_)
print(f"Test accuracy: {joint_search.score(X_test, y_test):.3f}")

Best joint configuration: {'model__C': 1, 'reduce': PCA(random_state=42), 'reduce__n_components': 10}
Test accuracy: 0.979


In [4]:
from sklearn.ensemble import RandomForestClassifier

model_selector = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=3000))])
model_grid = [
    {"scale": [StandardScaler()], "clf": [LogisticRegression(max_iter=3000)], "clf__C": [0.1, 1, 10]},
    {"scale": ["passthrough"], "clf": [RandomForestClassifier(random_state=42)], "clf__n_estimators": [80, 150], "clf__max_depth": [None, 5]},
]
model_search = GridSearchCV(model_selector, model_grid, cv=cv, scoring="accuracy", n_jobs=1).fit(X_train, y_train)
print("Selected estimator:", model_search.best_estimator_.named_steps["clf"])
print(f"Held-out accuracy: {model_search.score(X_test, y_test):.3f}")

Selected estimator: LogisticRegression(C=1, max_iter=3000)
Held-out accuracy: 0.986


## Key Takeaways
- A pipeline is the default safe unit for preprocessing plus modeling.
- Cross-validation refits every pipeline step within each training fold.
- Use `step__parameter` names for grid search.
- `named_steps` exposes fitted transformers and models for interpretation.
- Searches can compare preprocessing strategies and estimator families without leakage.